<a href="https://colab.research.google.com/github/HarshTikone/LoRA-bench/blob/main/notebooks/finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LoRA Bench — Day 2: LoRA/QLoRA Fine-Tuning

Fine-tunes `Qwen/Qwen2.5-Coder-1.5B-Instruct` with QLoRA on the CVE
fix-diff dataset prepared by this repo's Day 1 data-prep pipeline, runs a
small LoRA rank sweep, then does the full fine-tune with the winning
config and saves the adapter. It defaults to a two-step **T4 smoke
test**; set `RUN_MODE = "full"` only for the reviewed full run.

Designed for the **free Colab T4 tier** — no paid tier, no external paid
API. See the repo's `README.md`/`ADR.md` for the full project context;
this notebook is one stage of `data prep -> fine-tune -> quantize ->
benchmark -> report` (Day 3/4 add the rest, in this same notebook).

**Rough total runtime estimate on a T4** (data prep + sweep + full
fine-tune + a quick qualitative check): well under an hour. This is an
estimate, not a measurement — this notebook can't be run outside Colab to
verify it, so treat the first run as the actual measurement.


## Before you start

1. **Runtime > Change runtime type > T4 GPU**, then re-run from the top.
2. **Optional**: add an `HF_TOKEN` secret (key icon, left sidebar) — a
   free, read-scope Hugging Face token. Neither the dataset
   (`hitoshura25/cvefixes`) nor the base model is gated, so this only
   raises Hub rate limits; the notebook runs fine without it.
3. **The clone cell below needs this repo to be reachable at the URL you
   set in `REPO_URL`.** If it's private, either make it public or clone
   via a token (`https://<token>@github.com/...` — paste the token
   through Colab Secrets, never hardcode it in a cell).
4. Run cells top to bottom. Nothing here needs manual babysitting once
   started, beyond watching for the sweep/training progress bars.


In [ ]:
import subprocess
import time

import torch

RUN_STARTED_AT = time.time()
GPU_INFO = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True).stdout
print(GPU_INFO)

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime > Change runtime type > T4 GPU, "
    "then Runtime > Restart session, then re-run from the top."
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
import importlib
import os
import sys
from pathlib import Path

# Update if you forked/renamed the repo, or see "Before you start" above
# for how to clone a private repo.
RUN_MODE = "smoke"  # Accepted values: "smoke", "full". Keep the committed default.
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError(f"RUN_MODE must be 'smoke' or 'full', got {RUN_MODE!r}")
SMOKE_TEST = RUN_MODE == "smoke"
REPO_URL = "https://github.com/HarshTikone/LoRA-bench.git"
REPO_DIR = "/content/lora-bench"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

# Repo-side deps (datasets, huggingface_hub, python-dotenv, PyYAML) come
# from pyproject.toml via the editable install. GPU-side deps are listed
# separately in requirements-colab.txt and installed explicitly here --
# torch itself is deliberately NOT reinstalled, to avoid fighting Colab's
# preinstalled CUDA-matched build. jinja2 is listed explicitly even though
# it happens to already be present (pulled in transitively by Colab's
# preinstalled torch) -- tokenizer.apply_chat_template needs it directly,
# and this notebook shouldn't depend on that staying true by accident.
%pip install -q -e .
%pip install -q -U transformers peft bitsandbytes accelerate jinja2
importlib.invalidate_caches()
# Editable installs expose src/ through a .pth file. A running Colab kernel
# may not process a newly-created .pth file until restart, so make this
# checkout importable immediately and keep the path assertion below.
source_path = (Path(REPO_DIR) / "src").resolve()
if str(source_path) not in sys.path:
    sys.path.insert(0, str(source_path))
lora_bench = importlib.import_module("lora_bench")

package_path = Path(lora_bench.__file__).resolve()
assert package_path.is_relative_to(Path(REPO_DIR).resolve()), (
    f"lora_bench imported from {package_path}, not the cloned repo {REPO_DIR}"
)
print(f"lora_bench {lora_bench.__version__} imported from {package_path}")
print(f"RUN_MODE={RUN_MODE}")

In [ ]:
import os

from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata

    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
else:
    print(
        "No HF_TOKEN found (Colab Secrets panel, key icon in the left sidebar). "
        "Continuing without it -- the dataset and base model are both public, "
        "so this only affects Hub rate limits, not whether this notebook runs."
    )

## 1. Data prep

Runs the exact same, already-unit-tested pipeline from Day 1
(`src/lora_bench/data/cvefixes.py`) against the live, pinned dataset
revision (see `ADR.md`'s ADR-0002) -- nothing about this step is
Colab-specific, it's just running repo code that needs network access
this environment doesn't restrict.


In [ ]:
!{sys.executable} -m lora_bench.data.cvefixes --config configs/default.yaml --out-dir data/processed

In [ ]:
import json
import platform

with open("data/processed/manifest.json") as f:
    manifest = json.load(f)

RUN_METADATA = {
    "run_mode": RUN_MODE,
    "smoke_test": SMOKE_TEST,
    "python": sys.version,
    "platform": platform.platform(),
    "git_sha": subprocess.run(
        ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
    ).stdout.strip(),
    "gpu_info": GPU_INFO,
}
print(json.dumps(manifest, indent=2))
print(json.dumps(RUN_METADATA, indent=2))

## 2. Tokenizer & tokenized datasets

The tested package-level preprocessing renders the generation prompt and
full conversation separately, validates their exact text boundary, and
tokenizes the assistant suffix separately so BPE boundary merging cannot
change the generation prompt. Prompt labels are masked with `-100`. Examples
over `max_seq_len` are dropped and counted—fixed-code responses are never
truncated. Smoke mode limits training to 64 retained examples and validation
to 16.


In [ ]:
from transformers import AutoTokenizer

from lora_bench.config import load_config

cfg = load_config("configs/default.yaml")
BASE_MODEL = cfg.model.base_model
MAX_SEQ_LEN = cfg.model.max_seq_len

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"base model: {BASE_MODEL}")
print(f"max_seq_len: {MAX_SEQ_LEN}")
print(f"pad_token: {tokenizer.pad_token!r}")

In [ ]:
from collections import Counter

from datasets import Dataset

from lora_bench.data.cvefixes import read_jsonl
from lora_bench.training import (
    IGNORE_INDEX,
    TokenizationDropReason,
    tokenize_training_example,
)


def assert_disjoint_cves(*splits):
    id_sets = [{example.cve_id for example in split} for split in splits]
    for i, ids in enumerate(id_sets):
        for other in id_sets[i + 1 :]:
            assert ids.isdisjoint(other), "CVE leakage detected across data splits"


def prepare_split(examples, limit=None):
    records = []
    counts = Counter({reason.value: 0 for reason in TokenizationDropReason})
    eligible = 0
    for example in examples:
        outcome = tokenize_training_example(example, tokenizer, MAX_SEQ_LEN)
        if not outcome.kept:
            counts[outcome.drop_reason.value] += 1
            continue
        assert any(label != IGNORE_INDEX for label in outcome.record["labels"])
        eligible += 1
        if limit is None or len(records) < limit:
            records.append(outcome.record)
    assert records, "No examples survived exact tokenization checks"
    supervised_per_record = [
        sum(label != IGNORE_INDEX for label in record["labels"]) for record in records
    ]
    assert min(supervised_per_record) > 0, "Every retained example needs assistant tokens"
    token_stats = {
        "retained_tokens": sum(len(record["input_ids"]) for record in records),
        "supervised_tokens": sum(supervised_per_record),
        "min_supervised_tokens_per_example": min(supervised_per_record),
        "max_supervised_tokens_per_example": max(supervised_per_record),
        "masked_tokens": sum(
            label == IGNORE_INDEX for record in records for label in record["labels"]
        ),
    }
    return Dataset.from_list(records), dict(counts), eligible, token_stats


train_examples = read_jsonl("data/processed/train.jsonl")
val_examples_all = read_jsonl("data/processed/val.jsonl")
test_examples = read_jsonl("data/processed/test.jsonl")
assert_disjoint_cves(train_examples, val_examples_all, test_examples)
train_ds, train_token_drops, train_eligible, train_token_stats = prepare_split(
    train_examples, 64 if SMOKE_TEST else None
)
val_ds, val_token_drops, val_eligible, val_token_stats = prepare_split(
    val_examples_all, 16 if SMOKE_TEST else None
)
PREPROCESSING_COUNTERS = {
    "run_mode": RUN_MODE,
    "train": {
        "source": len(train_examples),
        "eligible": train_eligible,
        "retained": len(train_ds),
        "dropped": train_token_drops,
        **train_token_stats,
    },
    "val": {
        "source": len(val_examples_all),
        "eligible": val_eligible,
        "retained": len(val_ds),
        "dropped": val_token_drops,
        **val_token_stats,
    },
}
assert not SMOKE_TEST or len(train_ds) <= 64
assert len(val_ds) > 0
with open("/content/preprocessing_counters.json", "w") as f:
    json.dump(PREPROCESSING_COUNTERS, f, indent=2)
print(json.dumps(PREPROCESSING_COUNTERS, indent=2))

## 3. Base model loading (4-bit QLoRA)

`load_base_model()` is a function, not a one-off cell, because the sweep
below needs a **fresh** quantized base model per candidate: reusing one
base model object across multiple `get_peft_model()` calls risks stacking
adapters instead of cleanly replacing one, which isn't worth the risk of
a subtle bug for the ~30-60s a full reload costs on a 1.5B model.


In [ ]:
import torch
from peft import prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# T4 is Turing-generation and doesn't support bf16 compute well; detect
# rather than hardcode, so this also works correctly on newer GPUs.
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
QUANTIZATION_METADATA = {
    "load_in_4bit": True,
    "quant_type": "nf4",
    "double_quant": True,
    "compute_dtype": str(COMPUTE_DTYPE),
}
print(f"compute dtype: {COMPUTE_DTYPE}")


def load_base_model(*, for_training=True):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
    )
    assert getattr(model, "is_loaded_in_4bit", False), "Base model is not loaded in 4-bit"
    if for_training:
        return prepare_model_for_kbit_training(model)
    return model

## 4. LoRA hyperparameter sweep

The past-"just a demo" checklist requires the *final* rank/hyperparameters
be a defended decision from an actual sweep, not just a logged default
(see `src/lora_bench/config.py`'s `LoRAConfig` docstring). Three
candidates spanning rank 8/16/32 (alpha scaled proportionally, alpha =
2 * r, a common LoRA heuristic), each trained briefly (`PROBE_MAX_STEPS`)
and compared by validation loss — a short probe, not a claim that these
numbers are the fully-converged final loss for each rank.

Everything else (dropout, target_modules) is held fixed at
`configs/default.yaml`'s values so this isolates rank as the variable
under test.


In [ ]:
import gc
import math
import time

from peft import LoraConfig, TaskType, get_peft_model
from transformers import Trainer, TrainingArguments

from lora_bench.training import CompletionOnlyDataCollator

collator = CompletionOnlyDataCollator(tokenizer=tokenizer, pad_to_multiple_of=8)

SEED = 42
ALL_SWEEP_CANDIDATES = [
    {"r": 8, "lora_alpha": 16},
    {"r": 16, "lora_alpha": 32},
    {"r": 32, "lora_alpha": 64},
]
SWEEP_CANDIDATES = ALL_SWEEP_CANDIDATES[:1] if SMOKE_TEST else ALL_SWEEP_CANDIDATES
PROBE_MAX_STEPS = 2 if SMOKE_TEST else 50
PROBE_BATCH_SIZE = 1 if SMOKE_TEST else 4
PROBE_GRAD_ACCUM = 1 if SMOKE_TEST else 4
PROBE_HPARAMS = {
    "seed": SEED,
    "max_steps": PROBE_MAX_STEPS,
    "per_device_train_batch_size": PROBE_BATCH_SIZE,
    "gradient_accumulation_steps": PROBE_GRAD_ACCUM,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "lora_dropout": cfg.lora.dropout,
    "target_modules": cfg.lora.target_modules,
}
SMOKE_ADAPTER_DIR = "/content/lora_bench_smoke_adapter"


def run_probe(candidate, tag):
    started_at = time.time()
    torch.cuda.reset_peak_memory_stats()
    model = load_base_model()
    lora_config = LoraConfig(
        r=candidate["r"],
        lora_alpha=candidate["lora_alpha"],
        lora_dropout=cfg.lora.dropout,
        target_modules=cfg.lora.target_modules,
        task_type=TaskType.CAUSAL_LM,
        bias="none",
    )
    model = get_peft_model(model, lora_config)

    args = TrainingArguments(
        output_dir=f"/content/sweep_{tag}",
        per_device_train_batch_size=PROBE_BATCH_SIZE,
        gradient_accumulation_steps=PROBE_GRAD_ACCUM,
        max_steps=PROBE_MAX_STEPS,
        learning_rate=PROBE_HPARAMS["learning_rate"],
        lr_scheduler_type=PROBE_HPARAMS["lr_scheduler_type"],
        warmup_steps=max(1, int(0.03 * PROBE_MAX_STEPS)),
        logging_steps=1 if SMOKE_TEST else 10,
        save_strategy="no",
        eval_strategy="no",
        optim="paged_adamw_8bit",
        bf16=(COMPUTE_DTYPE == torch.bfloat16),
        fp16=(COMPUTE_DTYPE == torch.float16),
        report_to="none",
        seed=SEED,
        data_seed=SEED,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator)
    train_result = trainer.train()
    assert trainer.state.global_step == PROBE_MAX_STEPS, (
        f"Expected {PROBE_MAX_STEPS} optimizer steps, got {trainer.state.global_step}"
    )
    eval_metrics = trainer.evaluate(eval_dataset=val_ds)
    assert math.isfinite(train_result.metrics["train_loss"]), "Non-finite training loss"
    assert math.isfinite(eval_metrics["eval_loss"]), "Non-finite validation loss"
    if SMOKE_TEST:
        model.save_pretrained(SMOKE_ADAPTER_DIR)
        tokenizer.save_pretrained(SMOKE_ADAPTER_DIR)

    result = {
        "run_mode": RUN_MODE,
        "seed": SEED,
        "probe_hyperparameters": PROBE_HPARAMS,
        "optimizer_steps": trainer.state.global_step,
        "val_loss": eval_metrics["eval_loss"],
        "eval_metrics": eval_metrics,
        "train_metrics": train_result.metrics,
        "log_history": trainer.state.log_history,
        "elapsed_seconds": time.time() - started_at,
        "peak_vram_bytes": torch.cuda.max_memory_allocated(),
    }

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

    return result

In [ ]:
sweep_results = []
for candidate in SWEEP_CANDIDATES:
    tag = f"r{candidate['r']}"
    print(f"--- probing {tag} (alpha={candidate['lora_alpha']}) ---")
    probe_result = run_probe(candidate, tag)
    sweep_results.append({**candidate, **probe_result})
    print(f"{tag}: val_loss={probe_result['val_loss']:.4f}")

print()
print(json.dumps(sweep_results, indent=2))

In [ ]:
expected_candidates = 1 if SMOKE_TEST else 3
assert len(sweep_results) == expected_candidates
assert [result["r"] for result in sweep_results] == ([8] if SMOKE_TEST else [8, 16, 32])
assert all(result["optimizer_steps"] == PROBE_MAX_STEPS for result in sweep_results)
assert all(math.isfinite(result["val_loss"]) for result in sweep_results)
winner = min(sweep_results, key=lambda result: result["val_loss"])
SWEEP_REPORT = {
    "run_mode": RUN_MODE,
    "seed": SEED,
    "selection_rule": "lowest finite completion-only validation loss",
    "probe_hyperparameters": PROBE_HPARAMS,
    "candidates": sweep_results,
    "selected_winner": {
        "r": winner["r"],
        "lora_alpha": winner["lora_alpha"],
        "val_loss": winner["val_loss"],
    },
}
with open("/content/lora_sweep_results.json", "w") as f:
    json.dump(SWEEP_REPORT, f, indent=2)
print(
    f"Winning config: r={winner['r']}, lora_alpha={winner['lora_alpha']}, "
    f"val_loss={winner['val_loss']:.4f}"
)
print()
if SMOKE_TEST:
    print("Smoke mode validates the stack only; this is not a rank-sweep decision.")
else:
    print(
        "Return lora_sweep_results.json for ADR-0006's defended rank decision. "
        "Never report these numbers before they come from a real run."
    )

## 5. Full fine-tune with the winning config

Same setup as each sweep probe, but a full multi-epoch run instead of a
50-step probe, using whichever config `winner` above resolved to.


In [ ]:
FULL_TRAINING_RESULT = None
ADAPTER_DIR = SMOKE_ADAPTER_DIR if SMOKE_TEST else "/content/lora_bench_adapter"

if SMOKE_TEST:
    # run_probe already trained, saved, and unloaded the two-step adapter.
    print("Smoke adapter is ready for fresh-base reload.")
else:
    FULL_EPOCHS = 3
    FULL_BATCH_SIZE = 4
    FULL_GRAD_ACCUM = 4
    full_started_at = time.time()
    torch.cuda.reset_peak_memory_stats()

    model = load_base_model()
    lora_config = LoraConfig(
        r=winner["r"],
        lora_alpha=winner["lora_alpha"],
        lora_dropout=cfg.lora.dropout,
        target_modules=cfg.lora.target_modules,
        task_type=TaskType.CAUSAL_LM,
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    steps_per_epoch = max(1, math.ceil(len(train_ds) / (FULL_BATCH_SIZE * FULL_GRAD_ACCUM)))
    total_steps = steps_per_epoch * FULL_EPOCHS
    print(f"steps/epoch: {steps_per_epoch}  total steps: {total_steps}")

    full_args = TrainingArguments(
        output_dir="/content/lora_bench_finetune",
        per_device_train_batch_size=FULL_BATCH_SIZE,
        gradient_accumulation_steps=FULL_GRAD_ACCUM,
        num_train_epochs=FULL_EPOCHS,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=max(1, int(0.03 * total_steps)),
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        optim="paged_adamw_8bit",
        bf16=(COMPUTE_DTYPE == torch.bfloat16),
        fp16=(COMPUTE_DTYPE == torch.float16),
        report_to="none",
        seed=SEED,
        data_seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=full_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
    )
    train_result = trainer.train()
    final_eval_metrics = trainer.evaluate(eval_dataset=val_ds)
    assert trainer.state.global_step == total_steps, (
        f"Expected {total_steps} full-training steps, got {trainer.state.global_step}"
    )
    assert math.isfinite(train_result.metrics["train_loss"]), "Non-finite full train loss"
    assert math.isfinite(final_eval_metrics["eval_loss"]), "Non-finite full eval loss"

    FULL_TRAINING_RESULT = {
        "run_mode": RUN_MODE,
        "seed": SEED,
        "selection_rule": SWEEP_REPORT["selection_rule"],
        "selected_candidate": SWEEP_REPORT["selected_winner"],
        "epochs": FULL_EPOCHS,
        "per_device_train_batch_size": FULL_BATCH_SIZE,
        "gradient_accumulation_steps": FULL_GRAD_ACCUM,
        "learning_rate": 2e-4,
        "expected_optimizer_steps": total_steps,
        "completed_optimizer_steps": trainer.state.global_step,
        "train_metrics": train_result.metrics,
        "final_eval_metrics": final_eval_metrics,
        "best_validation_loss": trainer.state.best_metric,
        "log_history": trainer.state.log_history,
        "elapsed_seconds": time.time() - full_started_at,
        "peak_vram_bytes": torch.cuda.max_memory_allocated(),
    }
    assert math.isfinite(FULL_TRAINING_RESULT["best_validation_loss"])

    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    with open("/content/full_training_result.json", "w") as f:
        json.dump(FULL_TRAINING_RESULT, f, indent=2)
    print(json.dumps(FULL_TRAINING_RESULT, indent=2))

    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
from peft import PeftModel

assert Path(ADAPTER_DIR).is_dir(), f"Missing saved adapter: {ADAPTER_DIR}"
reload_started_at = time.time()
torch.cuda.reset_peak_memory_stats()
model = PeftModel.from_pretrained(load_base_model(for_training=False), ADAPTER_DIR)
model.eval()
RELOAD_METADATA = {
    "run_mode": RUN_MODE,
    "fresh_base_model": True,
    "base_loaded_in_4bit": bool(getattr(model.get_base_model(), "is_loaded_in_4bit", False)),
    "reload_elapsed_seconds": time.time() - reload_started_at,
    "reload_peak_vram_bytes": torch.cuda.max_memory_allocated(),
}
assert RELOAD_METADATA["base_loaded_in_4bit"]
print(f"Adapter + tokenizer reloaded from {ADAPTER_DIR} into a fresh 4-bit base model.")

## 6. Fresh-reload generation check

This is not Day 3's quality benchmark. It is a deterministic held-out
generation proving that the saved adapter can be loaded into a fresh 4-bit
base-model instance and used for inference.


In [ ]:
from lora_bench.data.cvefixes import to_chat_messages


def generate(gen_model, example, max_new_tokens=300):
    messages = to_chat_messages(example)[:1]  # user turn only, no ground-truth fix
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(gen_model.device)
    with torch.no_grad():
        output_ids = gen_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True)


torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
reload_example = val_examples_all[0]
max_new_tokens = 64 if SMOKE_TEST else 300
reload_output = generate(model, reload_example, max_new_tokens=max_new_tokens)
assert reload_output.strip(), "Reloaded adapter generated an empty response"
RELOAD_METADATA.update(
    {
        "reload_and_generation_elapsed_seconds": time.time() - reload_started_at,
        "reload_and_generation_peak_vram_bytes": torch.cuda.max_memory_allocated(),
    }
)
RELOADED_GENERATION = {
    **RELOAD_METADATA,
    "seed": SEED,
    "do_sample": False,
    "max_new_tokens": max_new_tokens,
    "cve_id": reload_example.cve_id,
    "input": reload_example.input,
    "ground_truth": reload_example.output,
    "generated": reload_output,
}
with open("/content/reloaded_generation.json", "w") as f:
    json.dump(RELOADED_GENERATION, f, indent=2)
print(json.dumps(RELOADED_GENERATION, indent=2))
del model
gc.collect()
torch.cuda.empty_cache()

## Next

- The committed default remains `RUN_MODE = "smoke"`.
- For the reviewed Day 2 run, change only that value to `"full"`, run
  all cells in a fresh T4 runtime, and privately return
  `lora_bench_training_artifacts.zip`. Its three probe losses and full
  training result will support ADR-0006's rank decision.
- Day 3 (not yet built): quantize the fine-tuned model to GGUF or AWQ, and
  add the repo-side (non-GPU, testable) benchmark harness cells to this
  same notebook -- quality, latency, memory, cost per 1K tokens, base vs.
  fine-tuned vs. quantized.


In [ ]:
import shutil
from pathlib import Path

from google.colab import files

from lora_bench.artifacts import sha256_file, write_checksums

RUN_METADATA.update(
    {
        "elapsed_seconds": time.time() - RUN_STARTED_AT,
        "preprocessing": PREPROCESSING_COUNTERS,
        "sweep_report": SWEEP_REPORT,
        "full_training_result": FULL_TRAINING_RESULT,
        "quantization": QUANTIZATION_METADATA,
        "reload": RELOAD_METADATA,
        "pip_freeze": subprocess.run(
            [sys.executable, "-m", "pip", "freeze"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.splitlines(),
    }
)
artifact_name = "lora_bench_smoke_artifacts" if SMOKE_TEST else "lora_bench_training_artifacts"
artifact_dir = Path("/content") / artifact_name
if artifact_dir.exists():
    shutil.rmtree(artifact_dir)
artifact_dir.mkdir()
shutil.copytree(ADAPTER_DIR, artifact_dir / "adapter")
ARTIFACT_MANIFEST = {
    **manifest,
    "artifact_schema_version": 1,
    "run_mode": RUN_MODE,
    "run_git_sha": RUN_METADATA["git_sha"],
}
with (artifact_dir / "manifest.json").open("w") as f:
    json.dump(ARTIFACT_MANIFEST, f, indent=2)
payload_sources = {
    "preprocessing_counters.json": Path("/content/preprocessing_counters.json"),
    "lora_sweep_results.json": Path("/content/lora_sweep_results.json"),
    "reloaded_generation.json": Path("/content/reloaded_generation.json"),
}
if not SMOKE_TEST:
    payload_sources["full_training_result.json"] = Path("/content/full_training_result.json")
for destination, source in payload_sources.items():
    assert source.is_file(), f"Missing required artifact payload: {source}"
    shutil.copy2(source, artifact_dir / destination)
with (artifact_dir / "run_metadata.json").open("w") as f:
    json.dump(RUN_METADATA, f, indent=2)
CHECKSUMS = write_checksums(artifact_dir, RUN_MODE)
assert all(
    sha256_file(artifact_dir / relative_path) == digest
    for relative_path, digest in CHECKSUMS["files"].items()
)
archive_path = shutil.make_archive(str(artifact_dir), "zip", artifact_dir)
print(f"Packaged {RUN_MODE} evidence: {archive_path}")
files.download(archive_path)